# 03 · Cadenas de Markov sobre las alertas de inventario

Reto 1 — Monitoreo de saldo de inventarios, FEMSA Salud Chile (FCV). Entregable 3.

La pregunta de fondo es qué tan probable es que un producto que hoy está bien termine estancado en los próximos meses. Cada alerta por partida (semáforo de antigüedad, riesgo por política de canje, estado general, rango de antigüedad, unidad de conversión) se trata como un estado que cambia de un mes al siguiente, y con los siete meses disponibles se estima una cadena de Markov de primer orden.

Todo el cálculo vive en `src/reto_ma2004b/markov.py`; aquí solo se usa y se interpreta. La app de Streamlit (`app/streamlit_app.py`) usa las mismas funciones para cualquier SKU.

Contenido:
1. La tabla completa y las secuencias por SKU
2. Estimación de la matriz de transición
3. Propiedades de las cadenas
4. Un SKU: estimación con pocos datos
5. Predicción: hacia dónde va el inventario
6. ¿Se sostiene el supuesto de Markov?
7. Sensibilidad: fechas sospechosas y segmentos
8. Resumen

Requisito: haber corrido `uv run reto-ma2004b` para tener `data/processed/powerbi/partidas.csv`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from reto_ma2004b import markov as M
from reto_ma2004b.indicadores import etiqueta_mes
from reto_ma2004b.pipeline import raiz_del_repo

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 30)

ROOT = raiz_del_repo()
FIGURAS = ROOT / "reports" / "figures"
FIGURAS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": False, "font.size": 10})
COLOR = {"VERDE": "#0CA30C", "ÁMBAR": "#FAB219", "ROJO": "#D03B3B"}

def guardar(fig, nombre):
    fig.savefig(FIGURAS / f"{nombre}.pdf", bbox_inches="tight", metadata={"CreationDate": None})

partidas = pd.read_csv(ROOT / "data" / "processed" / "powerbi" / "partidas.csv", parse_dates=["MES", "FULCOM"])
partidas.shape

## 1. La tabla completa y las secuencias por SKU

La tabla `partidas` ya une los siete meses: una fila por SKU y mes, con todas las alertas calculadas por el pipeline. Para Markov lo que importa es cuántos meses aparece cada SKU, porque cada par de meses consecutivos es una transición.

In [ ]:
meses_por_sku = partidas.groupby("CODIGO")["MES"].nunique()
print(f"SKU distintos: {len(meses_por_sku):,}")
meses_por_sku.value_counts().sort_index().rename("SKU").to_frame().T

La gran mayoría (13,788) aparece los siete meses. El resto entra o sale del inventario a mitad del periodo. Si un SKU falta un mes, no se cuenta una transición que "salte" ese mes; opcionalmente se puede agregar un estado `SIN INVENTARIO` para modelar las altas y bajas (sección 7).

Así se ve la secuencia de un SKU, igual que la tabla de ejemplo del entregable:

In [ ]:
ejemplo = "SKU_002583"
pd.concat({a: M.secuencias(partidas, a, ejemplo).iloc[0] for a in M.ALERTAS}, axis=1) \
  .rename(index=etiqueta_mes)

Este producto pasa por el ciclo completo: verde, ámbar al cumplir 90 días desde la última compra, rojo después de 180 días y de vuelta a verde cuando se vuelve a comprar. Ese último paso es importante para entender todo lo que sigue: `FULCOM` es la fecha de la **última** compra, así que cada resurtido reinicia la antigüedad.

Las secuencias más comunes del semáforo en días:

In [ ]:
sec = M.secuencias(partidas, "SEMAFORO_DIAS").dropna()
sec.astype(str).agg(" → ".join, axis=1).value_counts().head(10).rename("SKU").to_frame()

Dos patrones dominan: productos que se quedan verdes los siete meses (se resurten seguido) y productos que se quedan rojos los siete meses (inventario que no se mueve). La línea `VERDE ×5 → ROJO → VERDE` es el error de junio: la fecha de compra retrocede un año y en julio regresa.

## 2. Estimación de la matriz de transición

Para cada alerta se cuentan las transiciones $n_{ij}$ (SKU que estaban en el estado $i$ un mes y en $j$ el siguiente) y se estima

$$\hat p_{ij} = \frac{n_{ij}}{\sum_k n_{ik}},$$

que es el estimador de máxima verosimilitud de una cadena homogénea. Se excluyen las transiciones donde origen o destino tienen la fecha de compra sospechosa de junio, porque no son cambios reales del producto (en la sección 7 se ve qué pasa si no se excluyen).

In [ ]:
global_dias = M.estimar(partidas, "SEMAFORO_DIAS")
print(f"{global_dias.n_transiciones:,} transiciones de {global_dias.n_skus:,} SKU")
display(global_dias.conteos.astype(int))
global_dias.matriz

In [ ]:
def heatmap(ax, p, titulo, fmt="{:.1%}"):
    ax.imshow(p.fillna(0).to_numpy(), cmap="Blues", vmin=0, vmax=1)
    corto = lambda xs: [x.replace(" días", " d").replace("Más de ", "> ") for x in xs]
    ax.set_xticks(range(len(p.columns)), corto(p.columns), rotation=0, fontsize=8)
    ax.set_yticks(range(len(p.index)), corto(p.index), fontsize=8)
    for i in range(len(p.index)):
        for j in range(len(p.columns)):
            v = p.iat[i, j]
            ax.text(j, i, "—" if pd.isna(v) else fmt.format(v), ha="center", va="center", fontsize=8,
                    color="white" if (not pd.isna(v) and v > 0.55) else "#222")
    ax.set_xlabel("Mes siguiente"); ax.set_ylabel("Mes actual")
    ax.set_title(titulo, fontsize=10, loc="left")
    for s in ax.spines.values():
        s.set_visible(False)

fig, ax = plt.subplots(figsize=(4.6, 3.4))
heatmap(ax, global_dias.matriz, "Semáforo de antigüedad (días)")
guardar(fig, "markov_semaforo_dias")

Lectura: un SKU verde sigue verde el mes siguiente con 97.1 % de probabilidad y pasa a ámbar con 2.9 %; nunca salta directo a rojo (con las fechas sospechosas excluidas). Uno ámbar se queda ámbar 66 %, se vuelve rojo 23 % y regresa a verde 11 % (se resurtió). Uno rojo se queda rojo 98.7 % del tiempo: el estancamiento es muy persistente.

Las seis alertas a la vez:

In [ ]:
resultados = {a: M.estimar(partidas, a) for a in M.ALERTAS}
fig, axes = plt.subplots(2, 3, figsize=(13, 7.6))
for ax, (a, r) in zip(axes.flat, resultados.items()):
    titulo = {"SEMAFORO_DIAS": "Semáforo (días)", "SEMAFORO_MESES": "Semáforo (meses, Excel)"}.get(
        a, M.ALERTAS[a]["titulo"])
    heatmap(ax, r.matriz, titulo)
fig.tight_layout()
guardar(fig, "markov_seis_alertas")

## 3. Propiedades de las cadenas

Para cada matriz se revisa:

- **Clases comunicantes**: grupos de estados entre los que se puede ir y volver. Una clase es *recurrente* si no se puede salir de ella y *transitoria* si sí.
- **Irreducible**: una sola clase, es decir, desde cualquier estado se puede llegar a cualquier otro.
- **Periodo**: el mcd de las longitudes de los ciclos; si hay al menos un $p_{ii} > 0$ en una clase irreducible, el periodo es 1 (aperiódica).
- **Ergódica**: irreducible y aperiódica. En una cadena finita esto garantiza una distribución estacionaria única $\pi = \pi P$ a la que converge la cadena desde cualquier punto de partida.
- **Estados absorbentes**: $p_{ii} = 1$.

In [ ]:
filas = []
for a, r in resultados.items():
    pr = M.propiedades(r.matriz)
    pi = pr["estacionaria"]
    critico = M.ESTADO_CRITICO[a]
    hoy = M.distribucion(partidas, a, excluir_sospechosas=True)
    filas.append({
        "Alerta": M.ALERTAS[a]["titulo"].split(" (")[0],
        "Estados": len(r.estados),
        "Clases": len(pr["clases"]),
        "Irreducible": pr["irreducible"], "Aperiódica": pr["aperiodica"], "Ergódica": pr["ergodica"],
        "Absorbentes": ", ".join(pr["absorbentes"]) or "—",
        "Estado crítico": critico,
        "% hoy en crítico": hoy.get(critico, np.nan),
        "% largo plazo en crítico": pi[critico] if pi is not None else np.nan,
    })
propiedades = pd.DataFrame(filas)
propiedades

Cinco de las seis cadenas son **ergódicas**: irreducibles (siempre hay un camino de regreso, el resurtido) y aperiódicas (todos los estados tienen $p_{ii} > 0$). La de **unidad de conversión** no lo es: `REVISAR` es absorbente y `OK` casi también, porque la unidad de conversión es un atributo fijo del producto, no algo que cambie con el tiempo. Ahí Markov no aporta predicción; la alerta se atiende corrigiendo el catálogo.

La columna de largo plazo es la parte llamativa. Hoy alrededor de 23 % de los SKU está en rojo, pero si las probabilidades actuales se mantuvieran, el equilibrio sería cercano a 59 %. La razón está en la matriz: entrar a rojo es relativamente fácil (23 % desde ámbar) y salir es muy difícil (1.3 % por mes). La cadena acumula inventario estancado.

In [ ]:
pr = M.propiedades(global_dias.matriz)
pd.DataFrame({
    "Proporción hoy": M.distribucion(partidas, "SEMAFORO_DIAS", excluir_sospechosas=True),
    "Equilibrio (π)": pr["estacionaria"],
    "Meses medios para volver (1/π)": pr["tiempo_retorno"],
    "Meses medios hasta ROJO": M.tiempos_primer_paso(global_dias.matriz, "ROJO"),
    "Meses que dura una visita (1/(1−p_ii))": 1 / (1 - np.diag(global_dias.matriz)),
})

- Un producto verde tarda en promedio unos 55 meses en llegar a rojo, y uno ámbar unos 21: la mayoría se resurte antes.
- Una vez en rojo, la estancia esperada es de unos 79 meses ($1/(1-0.9873)$). En la práctica, lo que llega a rojo no sale solo; necesita una acción (canje, promoción, baja).

## 4. Un SKU: estimación con pocos datos

Con siete meses, un SKU tiene como máximo seis transiciones. La matriz de máxima verosimilitud de un solo producto casi siempre tiene renglones vacíos (estados que nunca visitó) o probabilidades de 0 y 1 que no son creíbles. Por eso, para un SKU se usa un estimador suavizado hacia la matriz global $q$:

$$\hat p^{(sku)}_{ij} = \frac{n_{ij} + \alpha\, q_{ij}}{n_i + \alpha}.$$

Es la media posterior con una previa de Dirichlet centrada en la global; $\alpha$ se lee como "cuántas transiciones de la población vale la previa". Con $\alpha = 0$ es la estimación propia del SKU; con $\alpha$ grande se acerca a la global.

In [ ]:
crudo = M.estimar(partidas, "SEMAFORO_DIAS", codigo=ejemplo)
suave = M.estimar(partidas, "SEMAFORO_DIAS", codigo=ejemplo, alfa=2)
print(f"{ejemplo}: {crudo.n_transiciones} transiciones")
display(crudo.conteos.astype(int))
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.4))
heatmap(axes[0], crudo.matriz, f"{ejemplo}: solo su historia (α = 0)")
heatmap(axes[1], suave.matriz, f"{ejemplo}: suavizada (α = 2)")
fig.tight_layout()
guardar(fig, "markov_sku_ejemplo")

Las matrices de las seis alertas de un SKU salen con una sola llamada (`M.estimar_sku`); la app de Streamlit hace lo mismo para el código que se elija.

In [ ]:
{a: M.resumen_propiedades(M.propiedades(r.matriz)) for a, r in M.estimar_sku(partidas, ejemplo, alfa=2).items()}

## 5. Predicción: hacia dónde va el inventario

La distribución dentro de $k$ meses es $x_{t+k} = x_t P^k$. Se parte de julio 2026 (último mes) y se proyecta tanto la proporción de SKU como la proporción del **valor** del inventario.

In [ ]:
hoy_skus = M.distribucion(partidas, "SEMAFORO_DIAS", excluir_sospechosas=True)
hoy_valor = M.distribucion(partidas, "SEMAFORO_DIAS", ponderar=True, excluir_sospechosas=True)
p_valor = M.estimar(partidas, "SEMAFORO_DIAS", ponderar=True).matriz
proy_skus = M.proyectar(global_dias.matriz, hoy_skus, 12)
proy_valor = M.proyectar(p_valor, hoy_valor, 12)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, proy, titulo in [(axes[0], proy_skus, "Proporción de SKU"), (axes[1], proy_valor, "Proporción del valor (CLP)")]:
    for e in proy.columns:
        ax.plot(proy.index, proy[e], color=COLOR[e], lw=2, marker="o", ms=3, label=e)
        dy = {"ÁMBAR": -6, "ROJO": 6}.get(e, 0) if proy is proy_valor else 0
        ax.annotate(f"{proy[e].iloc[-1]:.1%}", (proy.index[-1], proy[e].iloc[-1]), xytext=(4, dy),
                    textcoords="offset points", va="center", fontsize=8)
    ax.set_title(titulo, loc="left", fontsize=10)
    ax.set_xlabel("Meses desde julio 2026")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.grid(axis="y", color="#e6e6e6")
fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, ncol=3, loc="upper center",
           bbox_to_anchor=(0.5, 1.06))
fig.tight_layout()
guardar(fig, "markov_proyeccion")
pd.concat({"SKU": proy_skus.loc[[0, 3, 6, 12]], "Valor": proy_valor.loc[[0, 3, 6, 12]]}, axis=1)

In [ ]:
valor_total = partidas.loc[partidas["MES"] == partidas["MES"].max(), "VALORIZACION"].clip(lower=0).sum()
verde_hoy = partidas[(partidas["MES"] == partidas["MES"].max()) & (partidas["SEMAFORO_DIAS"] == "VERDE")]
p3 = M.potencia(p_valor, 3)
print(f"Valor del inventario en julio: {valor_total / 1e6:,.0f} millones CLP")
print(f"Del valor que hoy está verde, en 3 meses: " +
      ", ".join(f"{e} {p3.loc['VERDE', e]:.1%}" for e in p3.columns))
print(f"Valor que hoy está verde y estaría en rojo en 6 meses: "
      f"{verde_hoy['VALORIZACION'].sum() * M.potencia(p_valor, 6).loc['VERDE', 'ROJO'] / 1e6:,.0f} millones CLP")

Las dos vistas cuentan historias distintas. Por número de SKU, el rojo crece poco a poco. Por valor, casi todo el inventario está en verde: los productos caros rotan rápido y lo que se estanca son, sobre todo, productos de poco valor. Para la auditoría la segunda vista es la relevante, porque el riesgo de sobrevaluación se mide en pesos.

Para gestionar la caducidad no basta con el promedio: hace falta saber **qué** productos. `riesgo_por_sku` calcula, para cada SKU que hoy no está en rojo, la probabilidad de estar en rojo en 3 meses usando su propia matriz suavizada, y la multiplica por su valor:

In [ ]:
riesgo = M.riesgo_por_sku(partidas, "SEMAFORO_DIAS", pasos=3, alfa=2)
print(f"SKU evaluados: {len(riesgo):,}; valor esperado en rojo dentro de 3 meses: "
      f"{riesgo['VALOR_ESPERADO_DESTINO'].sum() / 1e6:,.0f} millones CLP")
riesgo.groupby("ESTADO_ACTUAL")["PROB_DESTINO"].describe()[["count", "mean", "50%", "max"]]

In [ ]:
riesgo.head(10)[["CODIGO", "NOMBRE_LINEA", "POLITICA_CANJE", "ESTADO_ACTUAL", "DIAS_ANTIG",
                 "VALORIZACION", "PROB_DESTINO", "VALOR_ESPERADO_DESTINO"]]

Los primeros lugares son productos ámbar de alto valor: tienen entre 91 y 180 días sin compra y, si no se resurten o se mueven, tienen cerca de 50 % de probabilidad de estar en rojo en tres meses. Es la lista de trabajo natural para el área comercial.

## 6. ¿Se sostiene el supuesto de Markov?

El modelo supone que la matriz es la misma todos los meses (homogeneidad). Se prueba con la razón de verosimilitud de Anderson y Goodman, comparando la matriz de cada par de meses con la global.

In [ ]:
M.prueba_homogeneidad(partidas, "SEMAFORO_DIAS")

In [ ]:
por_mes = M.matrices_por_mes(partidas, "SEMAFORO_DIAS")
etq = [f"{etiqueta_mes(m)[:3]}→{etiqueta_mes(m + pd.offsets.MonthBegin())[:3]}" for m in por_mes]
diag = pd.DataFrame([np.diag(q.fillna(0).to_numpy()) for q in por_mes.values()], index=etq,
                    columns=global_dias.estados)
amb_rojo = pd.Series([q.loc["ÁMBAR", "ROJO"] for q in por_mes.values()], index=etq)

fig, ax = plt.subplots(figsize=(7.2, 3.2))
for e in diag.columns:
    ax.plot(etq, diag[e], color=COLOR[e], lw=2, marker="o", ms=4, label=f"{e} → {e}")
ax.plot(etq, amb_rojo, color=COLOR["ROJO"], lw=1.5, ls="--", marker="o", ms=3, label="ÁMBAR → ROJO")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.grid(axis="y", color="#e6e6e6")
ax.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
ax.set_title("Probabilidades de transición estimadas para cada par de meses", loc="left", fontsize=10)
guardar(fig, "markov_homogeneidad")
pd.concat([diag, amb_rojo.rename("ÁMBAR→ROJO")], axis=1)

La homogeneidad se rechaza con claridad. Verde y rojo son estables mes a mes, pero el ámbar no: la probabilidad de pasar de ámbar a rojo va de casi 0 % a más de 60 %. La explicación es de calendario: el ámbar es una ventana de 90 días y lo que pase con él depende de cuándo se compraron los productos que en ese momento están ahí. Cuando hay un lote grande comprado el mismo día, todo el lote cruza a rojo el mismo mes.

¿Sirve entonces para predecir? La prueba práctica es fuera de muestra: estimar con enero–abril, tomar a los SKU de abril y proyectar mayo, junio y julio.

In [ ]:
bt = M.backtest(partidas, "SEMAFORO_DIAS", "2026-04-01")
tabla_bt = bt.pivot_table(index="ESTADO", columns="HORIZONTE", values=["PROYECTADO", "OBSERVADO"]) \
             .reindex(global_dias.estados)
print(f"Error absoluto medio: {bt['ERROR_PP'].abs().mean():.2f} pp; máximo: {bt['ERROR_PP'].abs().max():.2f} pp")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
for e in global_dias.estados:
    s = bt[bt["ESTADO"] == e]
    ax.plot(s["HORIZONTE"], s["OBSERVADO"], color=COLOR[e], lw=2, marker="o", label=f"{e} observado")
    ax.plot(s["HORIZONTE"], s["PROYECTADO"], color=COLOR[e], lw=1.5, ls="--", marker="x", label=f"{e} proyectado")
ax.set_xticks([1, 2, 3], ["May (1 m)", "Jun (2 m)", "Jul (3 m)"])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.grid(axis="y", color="#e6e6e6")
ax.legend(frameon=False, fontsize=7, ncol=3, loc="center", bbox_to_anchor=(0.5, 0.45))
ax.set_title("Prueba fuera de muestra: SKU de abril 2026", loc="left", fontsize=10)
guardar(fig, "markov_backtest")
tabla_bt

A nivel agregado el modelo funciona: el error es de uno a dos puntos porcentuales hasta tres meses adelante. La falta de homogeneidad se nota en el ámbar (subestima julio), que es justo el estado más sensible al calendario de compras. Para un horizonte corto y una lectura de cartera, la cadena homogénea es una buena aproximación; para afinar el ámbar haría falta un historial más largo o modelar la antigüedad exacta en días.

## 7. Sensibilidad: fechas sospechosas y segmentos

In [ ]:
con_error = M.estimar(partidas, "SEMAFORO_DIAS", excluir_sospechosas=False)
pd.concat({"Excluyendo fechas sospechosas": global_dias.matriz,
           "Sin excluir": con_error.matriz}, axis=1)

Sin excluir el error de junio aparece una transición verde → rojo de 0.9 % que no existe en la realidad (un producto no puede envejecer 180 días en un mes). Es la razón para dejar la exclusión activa por defecto.

Con el estado `SIN INVENTARIO` se ven también las altas y bajas de productos:

In [ ]:
M.estimar(partidas, "SEMAFORO_DIAS", incluir_ausencia=True).matriz

Un SKU rojo sale del inventario con 6 % de probabilidad al mes, mucho más que uno verde (0.4 %): el inventario estancado es el que termina dándose de baja.

Por política de canje y por línea:

In [ ]:
seg = {}
for pol in ["P0", "P1"]:
    seg[f"Política {pol}"] = M.estimar(partidas, "SEMAFORO_DIAS", filtros={"POLITICA_CANJE": pol}).matriz
seg_tabla = pd.concat({k: v.stack() for k, v in seg.items()}, axis=1)
seg_tabla.loc[[("VERDE", "ÁMBAR"), ("ÁMBAR", "ROJO"), ("ÁMBAR", "VERDE"), ("ROJO", "VERDE"), ("ROJO", "ROJO")]]

In [ ]:
lineas = []
for linea, _ in partidas.groupby("NOMBRE_LINEA"):
    r = M.estimar(partidas, "SEMAFORO_DIAS", filtros={"NOMBRE_LINEA": linea})
    pi = M.propiedades(r.matriz)["estacionaria"]
    lineas.append({"Línea": linea, "SKU": r.n_skus, "VERDE→ÁMBAR": r.matriz.loc["VERDE", "ÁMBAR"],
                   "ÁMBAR→ROJO": r.matriz.loc["ÁMBAR", "ROJO"], "ROJO→VERDE": r.matriz.loc["ROJO", "VERDE"],
                   "Rojo hoy": M.distribucion(partidas, "SEMAFORO_DIAS", filtros={"NOMBRE_LINEA": linea},
                                              excluir_sospechosas=True)["ROJO"],
                   "Rojo largo plazo": pi["ROJO"] if pi is not None else np.nan})
pd.DataFrame(lineas).sort_values("Rojo largo plazo", ascending=False)

Las diferencias entre segmentos son grandes y útiles para el negocio:

- Los productos **sin canje (P0)** pasan de verde a ámbar casi el doble de rápido que los que tienen canje (3.9 % contra 2.1 % al mes). Son justamente los que no se pueden devolver al proveedor, así que el riesgo económico se concentra ahí.
- **Farmacia** y **Alimentos** salen de rojo con más frecuencia (1.6–2.1 % al mes) y su equilibrio de largo plazo en rojo es bajo (27–34 %).
- **Cuidado del bebé**, **Distribuibles** y **Mercaderías varias** casi no salen de rojo; si nada cambia, la mayor parte de sus SKU terminaría estancada. Son las líneas donde conviene revisar el surtido y las compras.

## 8. Resumen

- Se estimaron cadenas de Markov de primer orden para seis alertas con 89 mil transiciones de 16 mil SKU (enero a julio 2026), excluyendo las fechas de compra sospechosas de junio.
- Cinco cadenas son ergódicas; la de unidad de conversión es reducible con un estado absorbente, porque es un atributo fijo del producto.
- El rojo es muy persistente (98.7 % se queda) y el equilibrio de largo plazo tiene más del doble de SKU en rojo que hoy. Por valor el panorama es mejor: el inventario caro rota y lo estancado es de poco valor.
- La matriz no es homogénea en el tiempo (sobre todo el ámbar), pero la proyección a uno, dos y tres meses tiene un error de 1 a 2 puntos porcentuales.
- Para un SKU, la estimación se suaviza hacia la matriz global; con eso se arma una lista de productos con mayor valor esperado en rojo a tres meses.